# Scientific Python: NumPy and Pandas
## 00-10 Review and biology
We moved from one DNA string (04), through collections (05), reusable functions (06), to files and saved reports (07). Today packages help with numerical arrays and tables. A gene is a DNA region involved in producing a functional product. A sample is material measured in an experiment; our sample IDs and values are entirely fictional. We invent expression-like measurements to practice tables, not real RNA-seq counts or normalized biological expression.

Table orientation: rows = genes; columns = fictional samples plus gene_id metadata. Values are nonnegative arbitrary units (AU). Gene identifiers are labels, not numbers to average. A blank cell means an unspecified missing measurement, not zero. The table does not encode groups, replicates, or a model for biological variation. Descriptive summaries characterize only these invented values; no significance or differential expression is inferred.

## 10-30 Environment
Follow [02_ENVIRONMENT.md](02_ENVIRONMENT.md) for the live setup/checkpoint. Use the same interpreter for pip, scripts and the notebook kernel. Do not put shell install commands inside Python code cells. Run this version cell after selecting the kernel.

In [ ]:
import sys
import numpy as np
import pandas as pd
print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 30-50 Arrays: build, inspect, calculate
NumPy supplies numerical arrays. `import numpy as np` gives the module a short conventional name; np.array constructs an array from a list. A one-dimensional array has one axis and a shape such as (3,). `dtype` describes its element type. Input: three invented lengths 2,4,6 mm. Algorithm: add 1 mm to each; double each; inspect index0; calculate the arithmetic mean. Expected [3 5 7], [4 8 12], first2 and mean4. Task 1: try [1,3,5] and predict first and mean.

In [ ]:
lengths = np.array([2, 4, 6])
print(lengths.shape, lengths.dtype)
print(lengths + 1)
print(lengths * 2)
print(lengths[0])
print(lengths[1:])
print(lengths.mean())
print(lengths.sum(), lengths.min(), lengths.max())

Python lists and arrays use the same operators differently. For lists, + joins lists and * repeats items. For numeric arrays, equal-sized array + array adds corresponding elements and * scalar multiplies each element. An ordinary list + 1 would raise TypeError. Here array + 1 adds a scalar to every element; no advanced array-shape rules are needed. Task 2: predict all four results before running.

In [ ]:
plain = [2, 4, 6]
array = np.array(plain)
print(plain + plain)
print(plain * 2)
print(array + array)
print(array * 2)

## 50-75 Series and DataFrames
A Series is one labeled column; a DataFrame is a rectangular table of labeled columns sharing a row index. A Python dictionary maps keys to values and need not be rectangular; a DataFrame organizes columns with equal row count and gives table operations. We build from a dictionary of lists, connecting Session 05.

The index labels default to 0,1,2. A gene_id column is separate metadata unless explicitly made the index; today we keep it as a normal column so it survives filtering. Series labels can also be gene IDs. Expected Series Gene_X=2, Gene_Y=4; frame shape(2,2). Task 3: add one fictional gene consistently to both lists.

In [ ]:
signal = pd.Series([2.0, 4.0], index=["Gene_X", "Gene_Y"], name="AU")
print(signal)
small = pd.DataFrame({"gene_id": ["Gene_X", "Gene_Y"], "Sample_X": [2.0, 4.0]})
print(small)
print(small.shape)

Read CSV with Pandas after learning standard-library CSV. `head()` shows first rows; `info()` prints a structure/missing-count report and returns None, so call it directly. `shape` gives rows/columns; dtypes shows each column's type. Depending on Pandas version, text types may display differently; IDs remain text. Expected (4,4), numeric A/C, floating B because it has a blank. The full file and field meanings are in [datasets](../../datasets/README.md).

This path cell accepts course root or this notebook folder as cwd. Expected first rows: Gene_A 10/12/14, Gene_B 18/20/22, Gene_C 4/NaN/8, Gene_D 0/0/0.

In [ ]:
from pathlib import Path
course_root = Path.cwd()
if not (course_root / "datasets").is_dir():
    course_root = Path.cwd().parents[1]
data_path = course_root / "datasets" / "session-08" / "expression.csv"
table = pd.read_csv(data_path)
print(table.head())
table.info()
print(table.shape)
print(table.dtypes)

One bracket selects a column (Series); a list inside brackets selects several columns (DataFrame). `loc` selects row/column labels; `iloc` selects positions. Our initial labels happen to equal positions, but filtering preserves old labels, so do not assume that later. Boolean conditions create a row mask. sort_values returns an ordered result unless saved back. Expected first gene Gene_A; selected filter IDs Gene_A,Gene_B; descending A order Gene_B,Gene_A,Gene_C,Gene_D. tolist converts one Series to a Python list for compact display. Task 4: filter Sample_A >=4 and sort ascending.

In [ ]:
print(table["Sample_A"])
print(table[["gene_id", "Sample_A"]])
print(table.loc[0, "gene_id"])
print(table.iloc[0, 1])
selected = table[table["Sample_A"] >= 10]
print(selected[["gene_id", "Sample_A"]])
print(table.sort_values("Sample_A", ascending=False)["gene_id"].tolist())

## 75-95 Descriptive statistics
Mean=sum/n; median=middle value after sorting (average the central two for even n); min/max are extremes. Standard deviation (SD) measures spread in the same AU units. Pandas std defaults to sample SD: square root of sum of squared deviations divided by n-1 (`ddof=1`). It needs at least two observed values. NumPy std defaults to n (`ddof=0`); choose explicitly when comparing. We teach calculation meaning, not inferential statistics.

Sample_A=[10,18,4,0], sum32, mean8, sorted[0,4,10,18], median7, min0, max18. Squared deviations 4+100+16+64=184; sample SD=sqrt(184/3)=7.83156 AU. Task 5: hand-check the mean and median first.

In [ ]:
values = table["Sample_A"]
print("Mean:", values.mean())
print("Median:", values.median())
print("Min/max:", values.min(), values.max())
print("Sample SD:", values.std())
print("NumPy sample SD:", np.std(np.array([10, 18, 4, 0]), ddof=1))

NaN means missing numeric data. `isna()` exposes missing positions; sum counts True entries. Pandas mean defaults to skipna=True, using only observed values. B's observed [12,20,0] has sum32 and mean32/3=10.6667 AU. This uses three values, while A uses four. Use count() to show the denominator. skipna=False yields NaN. Never silently fill a blank with zero or drop a row. Comparisons with NaN here do not pass a >= filter. NumPy mean propagates NaN by default. These are calculation conventions, not a justification for a missing-data model.

In [ ]:
print(table.isna())
print(table.isna().sum())
print("Observed B:", table["Sample_B"].count())
print("B observed mean:", table["Sample_B"].mean())
print("B no-skip mean:", table["Sample_B"].mean(skipna=False))
print("NumPy missing mean:", np.array([12.0, np.nan, 0.0]).mean())

## 95-110 Practical workflow
Task 6 (nine independent minutes): reuse the loaded table, inspect shape/dtypes, select gene_id and Sample_A, filter A>=10, inspect missing values, compute A mean/median/sample SD, sort A descending and write two sentences. Building blocks were taught above. Three minutes to compare/repair. Expected two filtered IDs Gene_A/Gene_B; one missing B measurement, A mean8 median7 sample SD7.83156. Keep IDs in the displayed result. The full script [02_DEMO.py](02_DEMO.py) is a separate after-attempt reference; run it from any working directory. Do not turn these differences into biological findings.

## 110-120 Documentation and assessment
Read a function's parameter descriptions before guessing its defaults. Find skipna and ddof in [Pandas std documentation](https://pandas.pydata.org/pandas-docs/version/2.2/reference/api/pandas.DataFrame.std.html), and compare [NumPy beginner examples](https://numpy.org/doc/2.2/user/absolute_beginners.html). Use the assessment; then read homework. Next session will add plots; no plotting package is needed today.

Common errors: wrong kernel -> reselect/restart; wrong path -> inspect cwd; averaging IDs -> select numeric sample column; using loc as a position -> inspect index; interpreting NaN as zero -> inspect isna/count; comparing NumPy/Pandas SD -> state ddof. Offline: use the expected tables/calculations and trace code; do not spend later blocks waiting for downloads.